#  Modélisation ML - Prédiction du Churn
**Projet : Customer Churn Analytics**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier

df = pd.read_csv('../../data/churn_data_clean.csv')
print('Shape:', df.shape)
df.head()

## 1. Préparation Train/Test

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape)
print('Test:', X_test.shape)

## 2. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print('--- Logistic Regression ---')
print(classification_report(y_test, y_pred_lr))
print('AUC-ROC:', roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]))

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('--- Random Forest ---')
print(classification_report(y_test, y_pred_rf))
print('AUC-ROC:', roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

## 4. XGBoost

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
print('--- XGBoost ---')
print(classification_report(y_test, y_pred_xgb))
print('AUC-ROC:', roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]))

## 5. Comparaison des modèles (ROC Curve)

In [ ]:
plt.figure(figsize=(10,6))
for model, name in [(lr,'Logistic Regression'), (rf,'Random Forest'), (xgb,'XGBoost')]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:,1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparaison ROC Curve')
plt.legend()
plt.savefig('../../python/outputs/roc_curve.png')
plt.show()

## 6. Feature Importance (XGBoost)

In [ ]:
feat_imp = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)[:15]
plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Top 15 Features Importantes (XGBoost)')
plt.savefig('../../python/outputs/feature_importance.png')
plt.show()